# Notebook 04: Baseline Features and Model

**Purpose:** Build the first honest, reproducible baseline.

Scope (intentionally minimal):
- Features: lagged values of the feature matrix (lags 1, 2, 3, 5, 10 days)
- Model: Ridge regression (one per target)
- Split: simple holdout — 80% train, 20% validation (by date_id)
- Metric: Spearman-Sharpe score on the validation set

**Goal:** A credible, reproducible score — not performance maximisation.
Any trained model should beat the zero-prediction and last-period baselines.

In [ ]:
import warnings
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

from src.data.load_data import load_train, load_labels, load_target_pairs, get_labels_for_lag
from src.evaluation.metric import spearman_sharpe_score, per_period_spearman, mean_row_pearson

warnings.filterwarnings('ignore')

train  = load_train()
labels = load_labels()
pairs  = load_target_pairs()

feature_cols = [c for c in train.columns if c != 'date_id']
print(f'Features: {len(feature_cols)}   Labels: {labels.shape[1] - 1}   Rows: {len(train)}')

## 1. Feature engineering — lagged feature values

In [ ]:
def build_lag_features(
    df: pd.DataFrame,
    feature_cols: list[str],
    lags: list[int] = [1, 2, 3, 5, 10],
) -> pd.DataFrame:
    """Build lagged feature matrix from the wide feature DataFrame.

    For each lag k, shift all feature columns down by k rows.
    The resulting NaN rows (first k rows) are kept so date_id alignment is preserved.
    """
    frames = []
    for lag in lags:
        shifted = df[feature_cols].shift(lag)
        shifted.columns = [f'{c}_lag{lag}' for c in feature_cols]
        frames.append(shifted)
    return pd.concat(frames, axis=1)


lag_features = build_lag_features(train, feature_cols, lags=[1, 2, 3, 5, 10])
lag_features.insert(0, 'date_id', train['date_id'].values)

print(f'Lag feature matrix shape: {lag_features.shape}')
print(f'NaN rows (from lagging): {lag_features.isnull().all(axis=1).sum()}')

## 2. Train / validation split

In [ ]:
HOLDOUT_FRACTION = 0.20
MIN_LAG          = 10  # rows dropped due to maximum lag

all_ids   = train['date_id'].values
T_total   = len(all_ids)
val_size  = int(T_total * HOLDOUT_FRACTION)
train_end = T_total - val_size

split_id  = all_ids[train_end]

X = lag_features.drop(columns='date_id').values
date_ids = lag_features['date_id'].values

# Drop rows where lag features are all NaN (first MIN_LAG rows)
valid_rows = ~np.all(np.isnan(X), axis=1)

# Split mask
train_mask = valid_rows & (date_ids < split_id)
val_mask   = valid_rows & (date_ids >= split_id)

print(f'Train rows: {train_mask.sum()}   Val rows: {val_mask.sum()}')
print(f'Split at date_id = {split_id}')

## 3. Naive baselines

In [ ]:
# Use lag=1 targets for the baseline evaluation
lag1_labels = get_labels_for_lag(labels, pairs, lag=1)
target_cols = [c for c in lag1_labels.columns if c != 'date_id']

# Align labels with lag_features by date_id
label_vals = lag1_labels.set_index('date_id')[target_cols]
y_true_val = label_vals.loc[date_ids[val_mask]]

# Zero prediction baseline
y_zero = pd.DataFrame(
    np.zeros_like(y_true_val.values),
    index=y_true_val.index,
    columns=y_true_val.columns,
)
score_zero = spearman_sharpe_score(y_zero, y_true_val, nan_policy='ignore')
print(f'Zero-prediction baseline  : {score_zero}')

# Last-period return (shift labels by 1 in validation)
y_lastperiod = label_vals.shift(1).loc[date_ids[val_mask]]
score_last = spearman_sharpe_score(y_lastperiod, y_true_val, nan_policy='ignore')
print(f'Last-period (momentum) baseline: {score_last:.4f}')

## 4. Ridge regression baseline

In [ ]:
X_train = X[train_mask]
X_val   = X[val_mask]

# Impute NaN with column mean (training set statistics only)
col_means = np.nanmean(X_train, axis=0)
nan_mask_train = np.isnan(X_train)
nan_mask_val   = np.isnan(X_val)
X_train_imp = X_train.copy()
X_val_imp   = X_val.copy()
X_train_imp[nan_mask_train] = np.take(col_means, np.where(nan_mask_train)[1])
X_val_imp[nan_mask_val]     = np.take(col_means, np.where(nan_mask_val)[1])

# Standardise (fit on training data only)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_imp)
X_val_sc   = scaler.transform(X_val_imp)

print(f'X_train: {X_train_sc.shape}   X_val: {X_val_sc.shape}')

In [ ]:
# Align labels to training and val indices
y_train = label_vals.loc[date_ids[train_mask]].values  # shape (T_train, N_targets)
y_val   = label_vals.loc[date_ids[val_mask]].values    # shape (T_val,   N_targets)

print(f'y_train: {y_train.shape}   y_val: {y_val.shape}')

In [ ]:
# Fit one Ridge model per target (multi-output can also be used)
# Replace NaN in y_train with 0 (targets with missing values contribute 0 signal)
y_train_imp = np.where(np.isnan(y_train), 0.0, y_train)

model = Ridge(alpha=1.0, fit_intercept=True)
model.fit(X_train_sc, y_train_imp)

y_pred_val = model.predict(X_val_sc)  # shape (T_val, N_targets)

y_pred_df = pd.DataFrame(
    y_pred_val,
    index=date_ids[val_mask],
    columns=target_cols,
)
y_true_df = pd.DataFrame(
    y_val,
    index=date_ids[val_mask],
    columns=target_cols,
)

print('Predictions generated.')

In [ ]:
score_ridge = spearman_sharpe_score(y_pred_df, y_true_df, nan_policy='ignore')
proxy_ridge = mean_row_pearson(y_pred_df, y_true_df, nan_policy='ignore')

print('=== Baseline Results (lag=1 targets, holdout validation) ===')
print(f'  Zero-prediction        : {score_zero}')
print(f'  Last-period return     : {score_last:.4f}')
print(f'  Ridge (alpha=1.0) [SS] : {score_ridge:.4f}   <- official metric')
print(f'  Ridge (alpha=1.0) [P]  : {proxy_ridge:.4f}   <- proxy only')

## 5. Per-period performance

In [ ]:
import matplotlib.pyplot as plt

per_period = per_period_spearman(y_pred_df, y_true_df)

plt.figure(figsize=(12, 3))
per_period.plot(alpha=0.7, label='Per-period Spearman')
plt.axhline(per_period.mean(), color='red', lw=1.2, linestyle='--',
            label=f'Mean = {per_period.mean():.4f}')
plt.axhline(0, color='black', lw=0.5)
plt.xlabel('date_id')
plt.ylabel('Spearman correlation')
plt.title('Ridge baseline — per-period Spearman correlation (validation set)')
plt.legend()
plt.tight_layout()
plt.show()

print(f'Mean : {per_period.mean():.4f}')
print(f'Std  : {per_period.std():.4f}')
print(f'Min  : {per_period.min():.4f}')
print(f'Max  : {per_period.max():.4f}')

## 6. Baseline result summary

| Model | Spearman-Sharpe | Pearson proxy |
|-------|-----------------|---------------|
| Zero prediction | — | — |
| Last-period return | TBD | — |
| Ridge (lag features, lag=1) | **TBD** | TBD |

Update this table after running the notebook.

**Next steps for Stage 3:**
- Tune Ridge alpha using walk-forward CV
- Engineer cross-asset spread features
- Evaluate all 4 lags, not just lag=1
- Try non-linear models (LightGBM)